# 56 — Đổi hình dạng: long ↔ wide

Dữ liệu finlens về ở **dạng long**: mỗi dòng một `(mã, phiên)`. Đó là dạng đúng
để lưu trữ và truyền tải, nhưng **sai** cho một số phép tính — ma trận tương
quan, backtest vectorised, hay bảng để người đọc nhìn đều cần **dạng wide**:
hàng là ngày, cột là mã.

Notebook này là bản đồ giữa hai thế giới đó.

| Từ | Sang | Dùng |
|---|---|---|
| long | wide | `pivot` (nghiêm ngặt) · `pivot_table` (có gộp) |
| wide | long | `melt` |
| cột ↔ tầng index | | `stack` · `unstack` |

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, heatmap, hom_nay, lui_ngay

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

danh_muc = client.meta.symbols(exchange="HOSE", kind="stock")
gia = client.eod.stock.ohlcv(danh_muc["symbol"].tolist()[:120], start=lui_ngay(HOM_NAY, nam=2))
gia = gia.sort_values(["symbol", "date"]).reset_index(drop=True)

print(f"pandas {pd.__version__} · dạng long: {len(gia):,} dòng × {gia.shape[1]} cột")
gia.head(3)

pandas 3.0.5 · dạng long: 58,468 dòng × 7 cột


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


,symbol,date,open,high,low,close,volume
0,AAA,2024-08-12,9.57,9.66,9.48,9.61,4558000.0
1,AAA,2024-08-13,9.66,9.75,9.57,9.66,3049900.0
2,AAA,2024-08-14,9.75,9.75,9.57,9.57,3149400.0


## 1 · Vì sao dữ liệu về ở dạng long

Ba lý do, và cả ba đều là lý do tốt:

1. **Thêm mã không đổi lược đồ.** Ở dạng wide, mỗi mã mới là một cột mới — bảng
   cơ sở dữ liệu phải `ALTER TABLE`.
2. **Không có ô rỗng.** Mã niêm yết năm 2024 đơn giản là không có dòng nào
   trước đó; ở dạng wide nó là một cột đầy `NaN`.
3. **Mỗi dòng tự mô tả.** Cột `symbol` nằm ngay trong dữ liệu, nên `groupby`,
   `merge`, ghi vào database đều thẳng thớm.

In [2]:
rong = gia.pivot(index="date", columns="symbol", values="close")
print(f"long: {gia.shape}  →  wide: {rong.shape}")
print(f"Ô có dữ liệu ở dạng wide: {rong.notna().sum().sum() / rong.size:.1%}")
print(f"→ {rong.isna().sum().sum():,} ô rỗng, phần lớn là mã niêm yết sau ngày đầu kỳ")

long: (58468, 7)  →  wide: (499, 119)
Ô có dữ liệu ở dạng wide: 98.5%
→ 913 ô rỗng, phần lớn là mã niêm yết sau ngày đầu kỳ


## 2 · `pivot` — long sang wide, nghiêm ngặt

⚠️ **pandas 3.0: `pivot` chỉ nhận tham số bằng từ khoá.** Cú pháp vị trí của
pandas 1.x đã bị bỏ.

In [3]:
try:
    gia.pivot("date", "symbol", "close")
except TypeError as e:
    print(f"Cú pháp vị trí → TypeError: {str(e)[:80]}")

print("\nCú pháp đúng: pivot(index=…, columns=…, values=…)")
print(f"  {gia.pivot(index='date', columns='symbol', values='close').shape}")

Cú pháp vị trí → TypeError: DataFrame.pivot() takes 1 positional argument but 4 were given

Cú pháp đúng: pivot(index=…, columns=…, values=…)
  (499, 119)


`pivot` **không gộp gì cả** — nó chỉ sắp xếp lại. Nếu một ô nhận nhiều giá trị,
nó ném lỗi thay vì đoán:

In [4]:
trung = pd.concat([gia.head(2), gia.head(2)])
try:
    trung.pivot(index="date", columns="symbol", values="close")
except ValueError as e:
    print(f"Có dòng trùng khoá → ValueError: {str(e)[:90]}")

print("\n→ Đây là tính năng, không phải hạn chế: nó bắt lỗi dữ liệu trùng cho bạn.")

Có dòng trùng khoá → ValueError: Index contains duplicate entries, cannot reshape

→ Đây là tính năng, không phải hạn chế: nó bắt lỗi dữ liệu trùng cho bạn.


### Nhiều `values` cùng lúc → cột `MultiIndex`

In [5]:
nhieu = gia.pivot(index="date", columns="symbol", values=["close", "volume"])
print(f"shape: {nhieu.shape} · cột {nhieu.columns.nlevels} tầng")
print(f"tầng 0: {nhieu.columns.get_level_values(0).unique().tolist()}")
print(f"tầng 1: {nhieu.columns.get_level_values(1).unique().tolist()[:5]} …")
print()
print("Lấy một tầng ra:")
print(f"  nhieu['close'].shape = {nhieu['close'].shape}")
print(f"  nhieu[('close', 'AAA')].shape = {nhieu[('close', 'AAA')].shape}")

shape: (499, 238) · cột 2 tầng
tầng 0: ['close', 'volume']
tầng 1: ['AAA', 'AAM', 'AAN', 'AAT', 'ABR'] …

Lấy một tầng ra:
  nhieu['close'].shape = (499, 119)
  nhieu[('close', 'AAA')].shape = (499,)


## 3 · `pivot_table` — khi cần gộp

Khác `pivot` ở đúng một điểm: nó **có `aggfunc`**, nên xử lý được nhiều giá trị
trên một ô.

In [6]:
theo_thang = gia.assign(thang=lambda d: d["date"].dt.to_period("M").astype(str)).pivot_table(
    index="thang",
    columns="symbol",
    values="close",
    aggfunc="mean",
)
print(f"Giá trung bình theo tháng: {theo_thang.shape}")
theo_thang.iloc[-4:, :6].round(2)

Giá trung bình theo tháng: (25, 119)


symbol,AAA,AAM,AAN,AAT,ABR,ABS
thang,,,,,,
2026-05,6.85,6.57,15.42,2.88,13.08,2.89
2026-06,6.90,6.38,15.77,2.81,12.10,2.97
2026-07,6.94,6.48,15.47,2.64,11.34,3.12
2026-08,7.25,7.00,15.41,2.32,11.68,3.07


⚠️ **`pivot_table` mặc định `aggfunc="mean"` và im lặng.** Nếu dữ liệu của bạn
lẽ ra không được trùng mà lại trùng, `pivot` sẽ báo lỗi còn `pivot_table` sẽ
**lấy trung bình của hai giá trị đáng ra chỉ có một** — không cảnh báo nào.

In [7]:
ban_trung = pd.concat([gia.head(3), gia.head(3).assign(close=lambda d: d["close"] * 2)])

print("Cùng dữ liệu trùng khoá:")
try:
    ban_trung.pivot(index="date", columns="symbol", values="close")
except ValueError:
    print("  pivot()       → ValueError, dừng lại")

kq = ban_trung.pivot_table(index="date", columns="symbol", values="close")
print(f"  pivot_table() → chạy tiếp, cho ra {kq.iloc[0, 0]:.2f}")
print(f"                  (trung bình của {gia['close'].iloc[0]:.2f} và {gia['close'].iloc[0] * 2:.2f})")
print()
print("→ Dùng `pivot` khi bạn TIN dữ liệu không trùng. Nó sẽ nói cho bạn biết nếu bạn sai.")

Cùng dữ liệu trùng khoá:
  pivot()       → ValueError, dừng lại
  pivot_table() → chạy tiếp, cho ra 14.41
                  (trung bình của 9.61 và 19.22)

→ Dùng `pivot` khi bạn TIN dữ liệu không trùng. Nó sẽ nói cho bạn biết nếu bạn sai.


### `pivot_table` với `margins` — thêm dòng/cột tổng

In [8]:
theo_nganh = (
    gia.merge(danh_muc[["symbol", "icb_name2"]], on="symbol")
    .assign(
        nam=lambda d: d["date"].dt.year,
        gtgd=lambda d: d["close"] * d["volume"] * 1_000 / 1e9,
    )
    .pivot_table(
        index="icb_name2",
        columns="nam",
        values="gtgd",
        aggfunc="sum",
        margins=True,
        margins_name="Tổng",
    )
)
print("Giá trị giao dịch theo ngành và năm (tỷ đồng):")
theo_nganh.round(0)

Giá trị giao dịch theo ngành và năm (tỷ đồng):


nam,2024,2025,2026,Tổng
icb_name2,,,,
Bán lẻ,5211.0,21356.0,13168.0,39735.0
Bảo hiểm,2840.0,10395.0,9548.0,22783.0
Bất động sản,43778.0,179120.0,59301.0,282199.0
Công nghệ Thông tin,8070.0,13903.0,3769.0,25742.0
Du lịch và Giải trí,122.0,489.0,76.0,687.0
Dầu khí,7296.0,23895.0,66218.0,97409.0
Dịch vụ tài chính,14513.0,76737.0,23374.0,114625.0
Hàng & Dịch vụ Công nghiệp,870.0,1772.0,354.0,2996.0
Hàng cá nhân & Gia dụng,314.0,820.0,276.0,1409.0


## 4 · `melt` — wide sang long

Phép ngược của `pivot`. Dùng khi bạn nhận dữ liệu từ Excel, từ một API cũ, hoặc
khi muốn đưa bảng wide vào `groupby`.

In [9]:
quay_lai = rong.reset_index().melt(
    id_vars="date",
    var_name="symbol",
    value_name="close",
)
print(f"wide {rong.shape} → long {quay_lai.shape}")
print(f"Trong đó {quay_lai['close'].isna().sum():,} dòng có close = NaN — chính là các ô rỗng của bảng wide")

sach = quay_lai.dropna(subset=["close"])
print(f"Sau dropna: {len(sach):,} dòng · frame gốc có {len(gia):,} dòng")
print(f"Khớp nhau: {len(sach) == len(gia)}")

wide (499, 119) → long (59381, 3)
Trong đó 913 dòng có close = NaN — chính là các ô rỗng của bảng wide
Sau dropna: 58,468 dòng · frame gốc có 58,468 dòng
Khớp nhau: True


`melt` là cách duy nhất để **đưa bảng wide vào `groupby`**, vì `groupby` cần
giá trị nhóm nằm trong một *cột*, không phải trong *tên cột*.

In [10]:
tom_tat = (
    sach.groupby("symbol", observed=True)["close"]
    .agg(["count", "mean", "std"])
    .nlargest(5, "mean")
    .round(2)
)
print("Năm mã giá cao nhất (tính từ bảng đã melt):")
print(tom_tat.to_string())

Năm mã giá cao nhất (tính từ bảng đã melt):
        count    mean    std
symbol                      
BMP       499  127.89  20.97
DHG       499   94.21   3.52
CTR       499   86.50  14.10
DGC       499   84.52  21.88
CTD       499   70.59   8.58


## 5 · `stack` và `unstack` — cột ↔ tầng index

Hai phép này làm việc trên **index**, không phải trên cột dữ liệu như `pivot`.

- `stack`: cột → tầng index trong cùng (wide → long)
- `unstack`: tầng index trong cùng → cột (long → wide)

In [11]:
chong = rong.stack()
print(f"rong.stack()   → {type(chong).__name__} {chong.shape}, index {chong.index.nlevels} tầng")
print(f"  tên các tầng: {chong.index.names}")
print()
print(chong.head(3).to_string())

rong.stack()   → Series (59381,), index 2 tầng
  tên các tầng: ['date', 'symbol']

date        symbol
2024-08-12  AAA       9.61
            AAM       6.73
            AAN        NaN


In [12]:
mo_ra = chong.unstack()
print(f"chong.unstack() → {mo_ra.shape}, khớp bảng wide gốc: {mo_ra.shape == rong.shape}")

chong.unstack() → (499, 119), khớp bảng wide gốc: True


### ⚠️ pandas 3.0 đổi hẳn hành vi `stack()`

Ở pandas 2.x, `stack()` **bỏ các ô `NaN`** theo mặc định, và bạn phải viết
`stack(dropna=False)` để giữ lại. pandas 3.0 dùng cài đặt mới: **giữ toàn bộ
ô**, và tham số `dropna` bị **xoá hẳn**.

In [13]:
print(f"Bảng wide có {rong.size:,} ô, trong đó {rong.isna().sum().sum():,} ô NaN")
print(f"stack() cho ra {len(chong):,} dòng")
print()
if len(chong) == rong.size:
    print("→ GIỮ toàn bộ ô, kể cả NaN. Đây là hành vi mới của pandas 3.0.")
    print(f"  Ở pandas 2.x, cùng lệnh này cho ra {rong.size - rong.isna().sum().sum():,} dòng.")

Bảng wide có 59,381 ô, trong đó 913 ô NaN
stack() cho ra 59,381 dòng

→ GIỮ toàn bộ ô, kể cả NaN. Đây là hành vi mới của pandas 3.0.
  Ở pandas 2.x, cùng lệnh này cho ra 58,468 dòng.


In [14]:
try:
    rong.stack(dropna=False)
except (ValueError, TypeError) as e:
    print(f"stack(dropna=False) → {type(e).__name__}:")
    print(f"  {str(e)[:150]}")

stack(dropna=False) → ValueError:
  dropna must be unspecified as the new implementation does not introduce rows of NA values. This argument will be removed in a future version of pandas


**Hệ quả di trú.** Code pandas 2.x dựa vào `stack()` để *vừa* đổi hình dạng
*vừa* lọc bỏ ô rỗng sẽ nhận về nhiều dòng hơn hẳn ở pandas 3.0 — và những dòng
thừa đó mang giá trị `NaN`, nên chúng lọt qua mọi phép tính rồi hiện ra ở cuối
dưới dạng trung bình bị lệch hoặc số đếm sai.

**Cách sửa: nói rõ ý định bằng `.dropna()`.**

In [15]:
print(f"stack()            → {len(rong.stack()):,} dòng")
print(f"stack().dropna()   → {len(rong.stack().dropna()):,} dòng   ← hành vi cũ, viết tường minh")
print()
tb_co_nan = rong.stack().mean()
tb_bo_nan = rong.stack().dropna().mean()
print(f"Trung bình giữ NaN : {tb_co_nan:.4f}")
print(f"Trung bình bỏ NaN  : {tb_bo_nan:.4f}")
print("→ ở đây bằng nhau vì mean() tự bỏ qua NaN, nhưng count() thì không:")
print(f"  count() giữ NaN: {rong.stack().count():,} · len(): {len(rong.stack()):,}  ← hai số khác nhau")

stack()            → 59,381 dòng
stack().dropna()   → 58,468 dòng   ← hành vi cũ, viết tường minh

Trung bình giữ NaN : 24.3209
Trung bình bỏ NaN  : 24.3209
→ ở đây bằng nhau vì mean() tự bỏ qua NaN, nhưng count() thì không:
  count() giữ NaN: 58,468 · len(): 59,381  ← hai số khác nhau


### `unstack` trên `MultiIndex` — chọn tầng nào để mở

Đây là chỗ `unstack` mạnh hơn `pivot`: nó chọn được tầng.

In [16]:
nhieu_tang = (
    gia.assign(nam=lambda d: d["date"].dt.year)
    .groupby(["symbol", "nam"], observed=True)["close"]
    .mean()
)
print(f"Series với index 2 tầng: {nhieu_tang.shape}, tầng {nhieu_tang.index.names}")
print()
print(f"unstack()      → mở tầng TRONG CÙNG (nam) thành cột: {nhieu_tang.unstack().shape}")
print(f"unstack('symbol') → mở tầng symbol thành cột       : {nhieu_tang.unstack('symbol').shape}")

Series với index 2 tầng: (354,), tầng ['symbol', 'nam']

unstack()      → mở tầng TRONG CÙNG (nam) thành cột: (119, 3)
unstack('symbol') → mở tầng symbol thành cột       : (3, 119)


In [17]:
nhieu_tang.unstack().head(5).round(2)

nam,2024,2025,2026
symbol,,,
AAA,8.52,7.54,7.05
AAM,6.85,6.75,6.40
AAN,NaN,NaN,15.57
AAT,3.53,3.35,2.92
ABR,10.70,12.07,12.48


## 6 · Dùng thật — ba việc chỉ làm được ở dạng wide

### 6.1 · Ma trận tương quan

`corr()` cần mỗi mã là một cột. Ở dạng long thì không tính được.

In [18]:
ls_rong = rong.pct_change()

# Chỉ lấy các mã có đủ lịch sử, nếu không tương quan tính trên vài chục điểm
du_lich_su = ls_rong.columns[ls_rong.notna().sum() >= 400]
tuong_quan = ls_rong[du_lich_su].corr()

print(f"{len(du_lich_su)} mã đủ lịch sử · ma trận {tuong_quan.shape}")

# Cặp tương quan cao nhất (bỏ đường chéo)
cap = tuong_quan.where(np.triu(np.ones(tuong_quan.shape), k=1).astype(bool)).stack()
print("\n8 cặp mã biến động giống nhau nhất:")
print(cap.nlargest(8).round(3).to_string())

116 mã đủ lịch sử · ma trận (116, 116)



8 cặp mã biến động giống nhau nhất:
symbol  symbol
DCM     DPM       0.842
AGR     CTS       0.770
        BSI       0.764
BSI     CTS       0.757
DIG     DXG       0.744
DXG     DXS       0.728
AAA     APH       0.711
BID     CTG       0.693


In [19]:
top20 = ls_rong[du_lich_su].std().nlargest(15).index
heatmap(
    tuong_quan.loc[top20, top20].round(2),
    tieu_de="Ma trận tương quan lợi suất — 15 mã biến động mạnh nhất",
    phu_de="Chỉ tính được sau khi đưa về dạng wide · thang phân kỳ khoá điểm giữa ở 0",
    nhan_mau="hệ số tương quan",
    dinh_dang_o="%{z:.2f}",
)

⚠️ Chú ý `np.triu(..., k=1)` ở trên: ma trận tương quan **đối xứng**, nên nếu
không lấy nửa tam giác thì mỗi cặp xuất hiện hai lần và đường chéo (luôn bằng
1,0) chiếm hết top.

In [20]:
khong_loc = tuong_quan.stack().nlargest(5)
print("Nếu không bỏ đường chéo:")
print(khong_loc.round(3).to_string())
print("→ toàn mã tự tương quan với chính nó")

Nếu không bỏ đường chéo:
symbol  symbol
AAA     AAA       1.0
AAM     AAM       1.0
AAT     AAT       1.0
ABR     ABR       1.0
ABS     ABS       1.0
→ toàn mã tự tương quan với chính nó


### 6.2 · Backtest vectorised

Notebook `34` dùng đúng hình dạng này: ma trận `ngày × mã` cho phép tính lợi
suất danh mục bằng một phép nhân ma trận thay vì một vòng lặp.

In [21]:
trong_so = pd.DataFrame(0.0, index=rong.index, columns=rong.columns)
chon = rong.columns[:10]
trong_so[chon] = 1 / len(chon)  # danh mục đều 10 mã

ls_danh_muc = (trong_so * ls_rong).sum(axis=1)
von = (1 + ls_danh_muc.fillna(0)).cumprod()

print(f"Danh mục {len(chon)} mã, phân bổ đều")
print(f"Lợi suất tích luỹ: {(von.iloc[-1] - 1) * 100:+.1f}%")
print(f"Tính bằng MỘT phép nhân ma trận trên {trong_so.shape} — không vòng lặp nào")

Danh mục 10 mã, phân bổ đều
Lợi suất tích luỹ: -3.8%
Tính bằng MỘT phép nhân ma trận trên (499, 119) — không vòng lặp nào


### 6.3 · Bảng cho người đọc

Người đọc muốn nhìn hàng là thời gian, cột là mã. Dạng long không đọc được.

In [22]:
gan_day = (
    gia[gia["date"] > gia["date"].max() - pd.Timedelta(days=10)]
    .pivot(index="date", columns="symbol", values="close")
    .iloc[:, :8]
)
print("Bảng giá 8 mã, các phiên gần nhất:")
gan_day.round(2)

Bảng giá 8 mã, các phiên gần nhất:


symbol,AAA,AAM,AAN,AAT,ABR,ABS,ABT,ACB
date,,,,,,,,
2026-08-03,7.20,6.76,15.20,2.38,11.00,3.05,50.5,22.55
2026-08-04,7.21,6.76,15.30,2.34,11.10,3.10,50.5,22.45
2026-08-05,7.16,6.76,15.30,2.23,11.10,3.09,50.6,22.45
2026-08-06,7.10,6.90,15.40,2.28,11.10,3.09,50.9,22.15
2026-08-07,7.15,7.07,15.55,2.34,11.10,3.04,50.5,22.40
2026-08-10,7.29,7.01,15.55,2.33,11.85,3.08,50.9,22.65
2026-08-11,7.46,7.30,15.50,2.35,12.65,3.05,50.5,22.65
2026-08-12,7.41,7.45,15.45,2.34,13.50,3.04,50.6,22.75


## 7 · Đi và về — có mất gì không?

Một phép kiểm đáng chạy khi bạn định đổi hình dạng nhiều lần trong pipeline.

In [23]:
di = gia.pivot(index="date", columns="symbol", values="close")
ve = (
    di.reset_index()
    .melt(id_vars="date", var_name="symbol", value_name="close")
    .dropna(subset=["close"])
    .sort_values(["symbol", "date"])
    .reset_index(drop=True)
)
goc = gia[["symbol", "date", "close"]].sort_values(["symbol", "date"]).reset_index(drop=True)

print(f"Gốc      : {goc.shape}")
print(f"Đi và về : {ve[['symbol', 'date', 'close']].shape}")
print(f"Giá trị khớp: {np.allclose(goc['close'], ve['close'])}")
print()
bang_kieu = pd.DataFrame(
    {"gốc": goc.dtypes.astype(str), "sau vòng đi–về": ve[goc.columns].dtypes.astype(str)}
)
bang_kieu["giữ nguyên"] = np.where(bang_kieu["gốc"] == bang_kieu["sau vòng đi–về"], "✓", "✗")
print(bang_kieu.to_string())

Gốc      : (58468, 3)
Đi và về : (58468, 3)
Giá trị khớp: True

                   gốc  sau vòng đi–về giữ nguyên
symbol          string          string          ✓
date    datetime64[ns]  datetime64[ns]          ✓
close          float64         float64          ✓


Với ba kiểu này thì đi và về an toàn — giá trị khớp, kiểu giữ nguyên.

⚠️ **Nhưng `category` thì không.** Cột `symbol` đi qua `pivot` để thành **tên
cột**, rồi `melt` dựng lại nó từ tên cột — và tên cột thì không mang thông tin
hạng mục. Nó quay về dạng chuỗi thường, lặng lẽ:

In [24]:
cat_goc = gia.astype({"symbol": "category"})
cat_ve = (
    cat_goc.pivot(index="date", columns="symbol", values="close")
    .reset_index()
    .melt(id_vars="date", var_name="symbol", value_name="close")
)
print(f"symbol trước pivot : {cat_goc['symbol'].dtype}")
print(f"symbol sau melt    : {cat_ve['symbol'].dtype}")
print(f"Bộ nhớ: {cat_goc['symbol'].memory_usage(deep=True) / 1024:,.0f} KB "
      f"→ {cat_ve['symbol'].memory_usage(deep=True) / 1024:,.0f} KB")

symbol trước pivot : category
symbol sau melt    : string
Bộ nhớ: 67 KB → 3,016 KB


## Tổng kết

| Bạn cần | Gọi |
|---|---|
| long → wide, dữ liệu không trùng | `df.pivot(index=…, columns=…, values=…)` |
| long → wide, cần gộp | `df.pivot_table(…, aggfunc=…)` |
| wide → long | `df.melt(id_vars=…, var_name=…, value_name=…)` |
| cột → tầng index | `df.stack()` — bỏ `NaN` trừ khi `dropna=False` |
| tầng index → cột | `s.unstack()` hoặc `s.unstack("tên_tầng")` |

**Bốn điều mang sang notebook sau:**

1. ⚠️ **`pivot` chỉ nhận từ khoá** ở pandas 3.0 — cú pháp vị trí ném `TypeError`.
2. **`pivot` báo lỗi khi trùng khoá, `pivot_table` lấy trung bình im lặng.**
   Dùng `pivot` khi bạn tin dữ liệu sạch — nó sẽ nói cho bạn biết nếu bạn sai.
3. ⚠️ **`stack()` ở pandas 3.0 GIỮ ô `NaN`** — ngược hẳn pandas 2.x — và tham
   số `dropna` đã bị xoá. Muốn hành vi cũ thì viết `.stack().dropna()`.
4. **Đổi hình dạng làm mất `category`** — và chỉ `category`. Cột `symbol` đi
   qua `pivot` → `melt` quay về chuỗi thường, bộ nhớ phình lại hàng chục lần.
   Nếu pipeline của bạn đổi hình dạng ở giữa, hãy `astype("category")` lại ở
   cuối.

---

**Tiếp theo:** [`57_ghep_du_lieu.ipynb`](57_ghep_du_lieu.ipynb) — `merge`,
`join`, `concat`, và cách bắt lỗi ghép sai bằng `validate=`.